# Submission 11 - Experiment 35A

This submission uses the winning pipeline from Experiment 35A: base identity target/frequency encoding plus digit-derived identity target/frequency features.

Local validation ROC-AUC: **0.945361**.

This improves on Experiment 33B (0.945331) by +0.000030.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

TARGET = 'Will_Buy_EV'
ID_COL = 'id'

numeric_cols = [
    'Age',
    'Annual_Income_USD',
    'Daily_Commute_km',
    'Number_of_Cars_Owned',
    'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work',
    'Environmental_Concern_Level'
]

categorical_cols = [
    'Gender',
    'City_Type',
    'Current_Car_Type',
    'Home_Charging_Possible',
    'Subsidy_Available',
    'Range_Anxiety_Level'
]

y = train[TARGET].astype(str).str.strip().map({'No': 0, 'Yes': 1}).astype(int)
X_train_raw = train.drop(columns=[TARGET, ID_COL]).copy()
X_test_raw = test.drop(columns=[ID_COL]).copy()

def identity_key(series):
    return series.astype('string').fillna('__MISSING__')

def make_smoothed_mapping(keys, target, smoothing=20):
    temp = pd.DataFrame({'key': identity_key(keys), 'target': target.values})
    stats = temp.groupby('key')['target'].agg(['count', 'mean'])
    global_mean = target.mean()
    mapping = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
    return mapping, global_mean

def add_full_identity_features(X_fit, X_apply, target, columns, prefix, smoothing=20):
    X_fit = X_fit.copy()
    X_apply = X_apply.copy()
    global_mean = target.mean()

    for col in columns:
        fit_keys = identity_key(X_fit[col])
        apply_keys = identity_key(X_apply[col])
        mapping, _ = make_smoothed_mapping(fit_keys, target, smoothing)

        X_fit[f'{prefix}_{col}_target'] = fit_keys.map(mapping).fillna(global_mean).astype(float)
        X_apply[f'{prefix}_{col}_target'] = apply_keys.map(mapping).fillna(global_mean).astype(float)

        freq = fit_keys.value_counts(normalize=True)
        X_fit[f'{prefix}_{col}_freq'] = fit_keys.map(freq).fillna(0).astype(float)
        X_apply[f'{prefix}_{col}_freq'] = apply_keys.map(freq).fillna(0).astype(float)

    return X_fit, X_apply

def add_oof_identity_features(X_fit, X_apply, target, columns, prefix, smoothing=20):
    X_fit = X_fit.copy()
    X_apply = X_apply.copy()
    global_mean = target.mean()
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    for col in columns:
        oof_target = np.zeros(len(X_fit), dtype=float)
        fit_keys_all = identity_key(X_fit[col])

        for fit_idx, val_idx in skf.split(X_fit, target):
            fit_keys = fit_keys_all.iloc[fit_idx]
            val_keys = fit_keys_all.iloc[val_idx]
            y_fit = target.iloc[fit_idx]

            mapping, _ = make_smoothed_mapping(fit_keys, y_fit, smoothing)
            oof_target[val_idx] = val_keys.map(mapping).fillna(global_mean).values

        X_fit[f'{prefix}_{col}_target'] = oof_target

        full_mapping, _ = make_smoothed_mapping(fit_keys_all, target, smoothing)
        apply_keys = identity_key(X_apply[col])
        X_apply[f'{prefix}_{col}_target'] = apply_keys.map(full_mapping).fillna(global_mean).astype(float)

        freq = fit_keys_all.value_counts(normalize=True)
        X_fit[f'{prefix}_{col}_freq'] = fit_keys_all.map(freq).fillna(0).astype(float)
        X_apply[f'{prefix}_{col}_freq'] = apply_keys.map(freq).fillna(0).astype(float)

    return X_fit, X_apply

digit_source_cols = [
    'Age',
    'Annual_Income_USD',
    'Daily_Commute_km',
    'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work'
]

def add_digit_identity_keys(X):
    X = X.copy()

    for col in digit_source_cols:
        values = pd.to_numeric(X[col], errors='coerce')
        integer_values = values.round().astype('Int64')
        text = integer_values.astype('string').fillna('__MISSING__')

        X[f'{col}_last2_key'] = text.map(lambda s: s if s == '__MISSING__' else s[-2:])
        X[f'{col}_last3_key'] = text.map(lambda s: s if s == '__MISSING__' else s[-3:])
        X[f'{col}_digit_sum_key'] = text.map(lambda s: s if s == '__MISSING__' else str(sum(int(ch) for ch in s if ch.isdigit())))
        X[f'{col}_first_last_key'] = text.map(lambda s: s if s == '__MISSING__' else s[0] + '_' + s[-1])
        X[f'{col}_count_last_key'] = text.map(lambda s: s if s == '__MISSING__' else str(len(s) if s.isdigit() else 0) + '_' + s[-1])

    return X

X_train = add_digit_identity_keys(X_train_raw)
X_test = add_digit_identity_keys(X_test_raw)

digit_identity_cols = []
for col in digit_source_cols:
    digit_identity_cols.extend([
        f'{col}_last2_key',
        f'{col}_last3_key',
        f'{col}_digit_sum_key',
        f'{col}_first_last_key',
        f'{col}_count_last_key'
    ])

X_train, X_test = add_oof_identity_features(
    X_train, X_test, y, numeric_cols, 'base', smoothing=20
)

X_train, X_test = add_oof_identity_features(
    X_train, X_test, y, digit_identity_cols, 'digit', smoothing=20
)

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print('Encoded train shape:', X_train_encoded.shape)
print('Encoded test shape:', X_test_encoded.shape)


In [ ]:
model = XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.04,
    min_child_weight=2,
    subsample=0.90,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    objective='binary:logistic',
    eval_metric='auc',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_encoded, y)

test_pred = model.predict_proba(X_test_encoded)[:, 1]

submission = pd.DataFrame({
    'id': test['id'],
    TARGET: test_pred
})

submission.to_csv('../submissions/submission_11.csv', index=False)

print('Submission 11 saved to ../submissions/submission_11.csv')
print(submission.head())
